# VO₂peak prediction demo with nested ElasticNet validation

Portfolio notebook based on the predictive-modelling part of my Master's thesis project.

**Important:** the original dataset cannot be shared publicly because it contains human-participant biomedical data.  
This notebook uses a **synthetic dataset** to demonstrate the modelling strategy without exposing private data.

## What this notebook demonstrates

- clinical-only, omics-only and combined predictive modelling;
- fold-wise preprocessing to avoid data leakage;
- nested ElasticNet model selection;
- out-of-fold evaluation;
- feature-selection stability;
- simple leave-one-feature-out sensitivity analysis for clinical variables.

In [ ]:
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import KFold, RepeatedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNetCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

FIGURE_DIR = PROJECT_ROOT / "figures" / "demo_outputs"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 120
pd.set_option("display.max_columns", 100)

## 1. Synthetic baseline dataset

The thesis prediction task used baseline metabolomic profiles and participant-level variables to estimate VO₂peak.

In this public notebook, we simulate:

- participant-level clinical variables;
- baseline metabolomic features;
- a synthetic VO₂peak target;
- weak metabolite effects compared with stronger clinical predictors.

This mirrors the modelling challenge without reproducing private data.

In [ ]:
def simulate_vo2peak_prediction_dataset(n_subjects=100, n_metabolites=52, random_state=42):
    rng = np.random.default_rng(random_state)

    subject_ids = [f"S{i:03d}" for i in range(1, n_subjects + 1)]

    age = np.clip(rng.normal(27, 8, n_subjects), 18, 63)
    sex_num = rng.binomial(1, 0.68, n_subjects)  # 1 = male, 0 = female
    bmi = np.clip(rng.normal(24.3, 3.5, n_subjects), 18, 36)
    relative_fat_mass = np.clip(13 + 0.9 * bmi - 5 * sex_num + rng.normal(0, 4, n_subjects), 5, 50)
    relative_fat_free_mass = 100 - relative_fat_mass
    endurance_weekly = rng.poisson(2.0 + 1.5 * sex_num, n_subjects)
    not_exhausted = rng.binomial(1, 0.23, n_subjects)

    clinical = pd.DataFrame({
        "Subject_ID": subject_ids,
        "Age": age,
        "Sex_num": sex_num,
        "BMI": bmi,
        "Relative_fat_mass": relative_fat_mass,
        "Relative_fat_free_mass": relative_fat_free_mass,
        "Endurance_weekly": endurance_weekly,
        "Not_exhausted_num": not_exhausted,
    }).set_index("Subject_ID")

    # Baseline metabolomic features with correlated latent structure.
    latent_fitness = (
        0.35 * (relative_fat_free_mass - relative_fat_free_mass.mean()) / relative_fat_free_mass.std()
        + 0.25 * (endurance_weekly - endurance_weekly.mean()) / endurance_weekly.std()
        - 0.20 * (age - age.mean()) / age.std()
        + rng.normal(0, 0.7, n_subjects)
    )

    metabolite_names = [f"Met_{i:02d}" for i in range(1, n_metabolites + 1)]
    omics = rng.normal(0, 1, size=(n_subjects, n_metabolites))

    # Only a small subset of metabolites carries weak signal.
    informative_idx = [1, 5, 9, 13, 18, 24, 31, 40]
    for j in informative_idx:
        omics[:, j] += 0.45 * latent_fitness + rng.normal(0, 0.25, n_subjects)

    X_omics = pd.DataFrame(omics, index=subject_ids, columns=metabolite_names)

    # Synthetic target: mostly clinical signal, modest metabolite contribution.
    omics_signal = X_omics.iloc[:, informative_idx].mean(axis=1).to_numpy()

    vo2peak = (
        52
        + 5.5 * sex_num
        - 0.35 * age
        - 0.65 * bmi
        - 0.22 * relative_fat_mass
        + 0.65 * endurance_weekly
        - 2.5 * not_exhausted
        + 1.2 * omics_signal
        + rng.normal(0, 4.5, n_subjects)
    )

    y = pd.Series(vo2peak, index=subject_ids, name="VO2peak")

    # Add a small amount of missingness to mimic real data.
    missing_mask = rng.random(X_omics.shape) < 0.02
    X_omics = X_omics.mask(missing_mask)

    clinical_missing = clinical.copy()
    for col in ["BMI", "Relative_fat_mass", "Endurance_weekly"]:
        idx = rng.choice(n_subjects, size=2, replace=False)
        clinical_missing.iloc[idx, clinical_missing.columns.get_loc(col)] = np.nan

    return clinical_missing, X_omics, y, informative_idx

X_clinical, X_omics, y, informative_idx = simulate_vo2peak_prediction_dataset()

print("Clinical matrix:", X_clinical.shape)
print("Omics matrix:", X_omics.shape)
print("Target:", y.shape)
display(X_clinical.head())
display(X_omics.head())

## 2. Modelling functions

The key principle is to avoid information leakage:

- imputation is fitted only on the training fold;
- scaling is fitted only on the training fold;
- ElasticNet hyperparameters are selected inside the training fold;
- final performance is computed on out-of-fold predictions.

In [ ]:
def build_elasticnet_pipeline(random_state=42):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", ElasticNetCV(
            l1_ratio=[0.1, 0.5, 0.9, 1.0],
            alphas=np.logspace(-3, 1, 40),
            cv=3,
            random_state=random_state,
            max_iter=50000
        ))
    ])


def regression_metrics(y_true, y_pred):
    return {
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }


def nested_elasticnet_oof_predictions(X, y, n_splits=4, random_state=42):
    """Return out-of-fold predictions and fold-level diagnostics."""
    outer_cv = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)

    oof_pred = pd.Series(index=y.index, dtype=float)
    fold_rows = []
    coefficients = []

    for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X), start=1):
        train_ids = y.index[train_idx]
        test_ids = y.index[test_idx]

        X_train = X.loc[train_ids]
        X_test = X.loc[test_ids]
        y_train = y.loc[train_ids]
        y_test = y.loc[test_ids]

        pipe = build_elasticnet_pipeline(random_state=random_state + fold)
        pipe.fit(X_train, y_train)

        pred = pipe.predict(X_test)
        oof_pred.loc[test_ids] = pred

        metrics = regression_metrics(y_test, pred)
        model = pipe.named_steps["model"]

        fold_rows.append({
            "Fold": fold,
            "n_train": len(train_ids),
            "n_test": len(test_ids),
            "alpha": float(model.alpha_),
            "l1_ratio": float(model.l1_ratio_),
            **{f"fold_{k}": v for k, v in metrics.items()}
        })

        coef = pd.Series(model.coef_, index=X.columns, name=f"Fold_{fold}")
        coefficients.append(coef)

    fold_results = pd.DataFrame(fold_rows)
    coef_table = pd.concat(coefficients, axis=1)

    return oof_pred, fold_results, coef_table


def compare_feature_sets(feature_sets, y, n_splits=4, random_state=42):
    all_predictions = {}
    fold_tables = {}
    coefficient_tables = {}
    summary_rows = []

    for name, X in feature_sets.items():
        pred, folds, coefs = nested_elasticnet_oof_predictions(
            X=X,
            y=y,
            n_splits=n_splits,
            random_state=random_state
        )

        all_predictions[name] = pred
        fold_tables[name] = folds
        coefficient_tables[name] = coefs

        metrics = regression_metrics(y, pred)
        summary_rows.append({"Model": name, **metrics})

    summary = pd.DataFrame(summary_rows).sort_values("R2", ascending=False)
    return summary, all_predictions, fold_tables, coefficient_tables


def stability_selection(X, y, n_splits=4, n_repeats=2, random_state=42):
    """Estimate how frequently each feature receives a non-zero ElasticNet coefficient."""
    cv = RepeatedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=random_state)
    counter = Counter()
    total_runs = n_splits * n_repeats

    for run, (train_idx, _) in enumerate(cv.split(X), start=1):
        train_ids = y.index[train_idx]
        pipe = build_elasticnet_pipeline(random_state=random_state + run)
        pipe.fit(X.loc[train_ids], y.loc[train_ids])
        coefs = pd.Series(pipe.named_steps["model"].coef_, index=X.columns)
        selected = coefs[coefs != 0].index
        counter.update(selected)

    stability = (
        pd.DataFrame(counter.items(), columns=["Feature", "Selected_count"])
        .assign(Selection_frequency=lambda d: d["Selected_count"] / total_runs)
        .sort_values("Selection_frequency", ascending=False)
        .reset_index(drop=True)
    )
    return stability

## 3. Clinical, omics and combined models

We compare three feature sets:

- clinical variables only;
- baseline metabolomics only;
- clinical + metabolomics.

In [ ]:
feature_sets = {
    "Clinical only": X_clinical,
    "Omics only": X_omics,
    "Clinical + omics": pd.concat([X_clinical, X_omics], axis=1),
}

summary, predictions, fold_tables, coefficient_tables = compare_feature_sets(
    feature_sets=feature_sets,
    y=y,
    n_splits=4,
    random_state=RANDOM_STATE
)

display(summary)

In [ ]:
# Fold-level diagnostics for the best model.
best_model_name = summary.iloc[0]["Model"]
print("Best synthetic-data model:", best_model_name)
display(fold_tables[best_model_name])

In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 4.2))
summary_plot = summary.set_index("Model")[["R2", "MAE", "RMSE"]]
summary_plot["R2"].plot(kind="bar", ax=ax)
ax.set_ylabel("Out-of-fold R²")
ax.set_title("Nested ElasticNet comparison")
ax.axhline(0, color="black", linewidth=0.8)
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "demo_vo2_model_comparison.png", bbox_inches="tight")
plt.show()

In [ ]:
# Observed vs predicted for each branch.
fig, axes = plt.subplots(1, 3, figsize=(12, 3.8), sharex=True, sharey=True)

y_min = min(y.min(), *(pred.min() for pred in predictions.values()))
y_max = max(y.max(), *(pred.max() for pred in predictions.values()))

for ax, (name, pred) in zip(axes, predictions.items()):
    metrics = regression_metrics(y, pred)
    ax.scatter(y, pred, alpha=0.75, s=35)
    ax.plot([y_min, y_max], [y_min, y_max], linestyle="--", color="black", linewidth=1)
    ax.set_title(f"{name}\nR²={metrics['R2']:.2f}, MAE={metrics['MAE']:.2f}")
    ax.set_xlabel("Observed VO₂peak")
    ax.grid(alpha=0.25)

axes[0].set_ylabel("Predicted VO₂peak")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "demo_vo2_observed_vs_predicted.png", bbox_inches="tight")
plt.show()

## 4. Feature-selection stability

ElasticNet can be used not only for prediction, but also to inspect which variables are repeatedly selected across resampling runs.

This should be interpreted cautiously: selection frequency is not proof of causality.  
It is a useful stability diagnostic, especially in small-sample biomedical datasets.

In [ ]:
stability_clinical = stability_selection(X_clinical, y, n_splits=4, n_repeats=2, random_state=RANDOM_STATE)
stability_omics = stability_selection(X_omics, y, n_splits=4, n_repeats=2, random_state=RANDOM_STATE)

print("Clinical feature stability")
display(stability_clinical)

print("Top omics feature stability")
display(stability_omics.head(12))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
top_features = stability_omics.head(12).copy()
ax.barh(top_features["Feature"][::-1], top_features["Selection_frequency"][::-1])
ax.set_xlabel("Selection frequency")
ax.set_title("Top metabolite selection frequencies")
ax.set_xlim(0, 1)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "demo_omics_selection_frequency.png", bbox_inches="tight")
plt.show()

## 5. Clinical leave-one-feature-out sensitivity analysis

The original prediction notebook also explored whether clinical-model performance depended strongly on specific variables.

Here we repeat a compact version: remove one clinical feature at a time and re-run the nested validation.

In [ ]:
loo_rows = []

# Baseline clinical model.
clinical_pred, _, _ = nested_elasticnet_oof_predictions(X_clinical, y, n_splits=4, random_state=RANDOM_STATE)
loo_rows.append({"Removed_feature": "None", **regression_metrics(y, clinical_pred)})

for feature in X_clinical.columns:
    X_reduced = X_clinical.drop(columns=[feature])
    pred, _, _ = nested_elasticnet_oof_predictions(X_reduced, y, n_splits=4, random_state=RANDOM_STATE)
    loo_rows.append({"Removed_feature": feature, **regression_metrics(y, pred)})

loo_results = pd.DataFrame(loo_rows)
display(loo_results.sort_values("R2", ascending=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
plot_df = loo_results.sort_values("R2", ascending=True)
ax.barh(plot_df["Removed_feature"], plot_df["R2"])
ax.set_xlabel("Out-of-fold R²")
ax.set_title("Clinical leave-one-feature-out sensitivity")
ax.axvline(loo_results.loc[loo_results["Removed_feature"] == "None", "R2"].iloc[0], color="black", linestyle="--", linewidth=1)
fig.tight_layout()
fig.savefig(FIGURE_DIR / "demo_clinical_leave_one_out.png", bbox_inches="tight")
plt.show()

## 6. Summary

This notebook demonstrates a leakage-aware prediction workflow for small biomedical datasets:

1. define clinically meaningful feature sets;
2. use fold-wise preprocessing;
3. evaluate models with out-of-fold predictions;
4. compare clinical and omics contributions;
5. inspect feature-selection stability and sensitivity.

The synthetic results should not be interpreted biologically.  
They are included only to make the portfolio notebook executable without private data.